# Student Placement Prediction + Segmentation

This notebook builds a student placement prediction workflow using academic, training, and profile data. It combines numeric scaling with TF-IDF for categorical text features and evaluates two classification models.

## Project goals

- Load and inspect the student placement dataset
- Convert boolean-style and categorical fields into numeric values
- Apply TF-IDF to grouped categorical text fields
- Train and compare Logistic Regression and Random Forest classifiers
- Evaluate model accuracy and classification performance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

df = pd.read_csv('Sample.csv')
print('Dataset shape:', df.shape)

display(df.head())

print('\nMissing values per column:')
print(df.isna().sum())

In [ ]:
# The next step is label encoding for boolean and categorical columns

# This cell has been replaced with preprocessing steps in the next cell.

## Preprocessing and label encoding

Convert Y/N fields and other categorical columns into numeric format before modeling.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_columns = [
    'Internships(Y/N)',
    'Gender',
    'Training(Y/N)',
    'Backlog in 5th sem',
    'Innovative Project(Y/N)',
    'Technical Course(Y/N)',
    'Placement(Y/N)?'
]

for col in label_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

print('Label encoding complete.')
print(df[label_columns].head())

In [ ]:
print('Encoded column data types:')
print(df[label_columns].dtypes)

display(df.head())

## Feature preparation

Create a combined text feature from categorical profile fields, keep numeric features for scaling, and define the target variable.

In [ ]:
df.drop(['Email', 'Name'], axis=1, inplace=True)

df['cat_text'] = (
    df['Stream'].fillna('') + ' ' +
    df['10th board'].fillna('') + ' ' +
    df['12th board'].fillna('')
)

X = df.drop(['Placement(Y/N)?', 'Stream', '10th board', '12th board'], axis=1)
y = df['Placement(Y/N)?']

print('Feature set shape:', X.shape)
print('Target distribution:')
print(y.value_counts())

## TF-IDF for categorical text and scaling for numeric features

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric features:', numeric_features)

vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(df['cat_text'].fillna(''))
print('TF-IDF features:', X_text.shape[1])

X_numeric = X[numeric_features].values
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X_numeric)

X_combined = hstack([X_numeric_scaled, X_text])

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.33, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## Model training and evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(criterion='entropy', n_estimators=250, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'report': classification_report(y_test, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred)
    }

for name, metrics in results.items():
    print('#' * 60)
    print(name)
    print('Accuracy:', round(metrics['accuracy'], 4))
    print('\nClassification report:\n', metrics['report'])

In [ ]:
best_model = max(results, key=lambda key: results[key]['accuracy'])
cm = results[best_model]['confusion_matrix']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model}')
plt.show()

print('Best model:', best_model)


## Summary

- The notebook loads student placement data, converts categorical values, and creates a combined text feature.
- TF-IDF is used for the categorical text fields while numeric features are scaled separately.
- Two classification models are trained and compared using accuracy and classification reports.

### Next improvements

- Add cross-validation and hyperparameter tuning
- Explore additional text or resume-based features
- Use feature importance analysis for model explainability